# 01 - Transcript Fetch

Explore the `youtube-transcript-api` library to understand:
- Data shape returned by the API
- Edge cases: missing transcripts, auto-generated captions, non-English, long/short videos
- What we need to handle in production

**Output:** Production-ready `fetch_transcript()` function → `src/transcript.py`

In [12]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable,
)
import json

ytt_api = YouTubeTranscriptApi()

## 1. Fetch a known video and inspect data shape

In [13]:
# Known video with English transcript — Rick Astley, Never Gonna Give You Up
transcript = ytt_api.fetch("dQw4w9WgXcQ")

print(f"Type: {type(transcript)}")
print(f"Language: {transcript.language}")
print(f"Language code: {transcript.language_code}")
print(f"Auto-generated: {transcript.is_generated}")
print(f"Video ID: {transcript.video_id}")
print(f"Snippet count: {len(transcript.snippets)}")
print(f"\n--- First 3 snippets ---")
for s in transcript.snippets[:3]:
    print(f"  Type: {type(s)}")
    print(f"  text: {s.text!r}")
    print(f"  start: {s.start} (type: {type(s.start).__name__})")
    print(f"  duration: {s.duration} (type: {type(s.duration).__name__})")
    print()

Type: <class 'youtube_transcript_api._transcripts.FetchedTranscript'>
Language: English
Language code: en
Auto-generated: False
Video ID: dQw4w9WgXcQ
Snippet count: 61

--- First 3 snippets ---
  Type: <class 'youtube_transcript_api._transcripts.FetchedTranscriptSnippet'>
  text: '[♪♪♪]'
  start: 1.36 (type: float)
  duration: 1.68 (type: float)

  Type: <class 'youtube_transcript_api._transcripts.FetchedTranscriptSnippet'>
  text: "♪ We're no strangers to love ♪"
  start: 18.64 (type: float)
  duration: 3.24 (type: float)

  Type: <class 'youtube_transcript_api._transcripts.FetchedTranscriptSnippet'>
  text: '♪ You know the rules\nand so do I ♪'
  start: 22.64 (type: float)
  duration: 4.32 (type: float)



In [14]:
last = transcript.snippets[-1]
video_duration = last.start + last.duration

print(f"--- Last snippet ---")
print(f"  text: {last.text!r}")
print(f"  start: {last.start}")
print(f"  duration: {last.duration}")
print(f"\nEstimated video duration: {video_duration:.1f}s ({video_duration/60:.1f} min)")
print(f"\n--- Gaps check (first 10) ---")
for i in range(min(10, len(transcript.snippets) - 1)):
    curr = transcript.snippets[i]
    nxt = transcript.snippets[i + 1]
    gap = nxt.start - (curr.start + curr.duration)
    print(f"  [{curr.start:.1f}–{curr.start + curr.duration:.1f}] → [{nxt.start:.1f}] gap: {gap:.2f}s")

--- Last snippet ---
  text: '♪ Never gonna tell a lie\nand hurt you ♪'
  start: 207.92
  duration: 3.4

Estimated video duration: 211.3s (3.5 min)

--- Gaps check (first 10) ---
  [1.4–3.0] → [18.6] gap: 15.60s
  [18.6–21.9] → [22.6] gap: 0.76s
  [22.6–27.0] → [27.0] gap: 0.08s
  [27.0–31.0] → [31.1] gap: 0.08s
  [31.1–35.1] → [35.2] gap: 0.08s
  [35.2–39.5] → [40.5] gap: 1.00s
  [40.5–42.9] → [43.0] gap: 0.08s
  [43.0–45.1] → [45.2] gap: 0.08s
  [45.2–47.1] → [47.3] gap: 0.24s
  [47.3–51.1] → [51.5] gap: 0.36s


## 2. Edge cases

Test different video types to understand what the API returns and what can go wrong.

In [15]:
test_videos = {
    "short_manual":      "dQw4w9WgXcQ",      # 3.5 min, manual captions
    "long_educational":  "aircAruvnKk",       # ~60 min, 3Blue1Brown
    "auto_generated":    "jNQXAC9IVRw",       # "Me at the zoo" — first YouTube video
    "non_english":       "g3jCAyPai2Y",        # Spanish — Bad Bunny
    "no_transcript":     "ZbZSe6N_BXs",       # Music — likely no transcript
}

for label, vid_id in test_videos.items():
    print(f"\n{'='*50}")
    print(f"Test: {label} ({vid_id})")
    print(f"{'='*50}")
    try:
        t = ytt_api.fetch(vid_id)
        last = t.snippets[-1]
        duration = last.start + last.duration
        print(f"  ✅ Language: {t.language} ({t.language_code})")
        print(f"  Auto-generated: {t.is_generated}")
        print(f"  Snippets: {len(t.snippets)}")
        print(f"  Duration: {duration:.0f}s ({duration/60:.1f} min)")
        print(f"  First text: {t.snippets[0].text[:60]!r}")
    except (TranscriptsDisabled, NoTranscriptFound) as e:
        print(f"  ❌ No transcript: {type(e).__name__}")
    except VideoUnavailable as e:
        print(f"  ❌ Video unavailable: {type(e).__name__}")
    except Exception as e:
        print(f"  ❌ Unexpected: {type(e).__name__}: {e}")


Test: short_manual (dQw4w9WgXcQ)
  ✅ Language: English (en)
  Auto-generated: False
  Snippets: 61
  Duration: 211s (3.5 min)
  First text: '[♪♪♪]'

Test: long_educational (aircAruvnKk)
  ✅ Language: English (en)
  Auto-generated: False
  Snippets: 286
  Duration: 1106s (18.4 min)
  First text: 'This is a 3.'

Test: auto_generated (jNQXAC9IVRw)
  ✅ Language: English (en)
  Auto-generated: False
  Snippets: 6
  Duration: 19s (0.3 min)
  First text: 'All right, so here we are, in front of the\nelephants'

Test: non_english (g3jCAyPai2Y)
  ❌ No transcript: TranscriptsDisabled

Test: no_transcript (ZbZSe6N_BXs)
  ✅ Language: English - en (en)
  Auto-generated: False
  Snippets: 75
  Duration: 237s (3.9 min)
  First text: '(upbeat music)'


In [16]:
# Check what's actually available for the videos that surprised us
check_videos = {
    "bad_bunny": "g3jCAyPai2Y",
    "me_at_zoo": "jNQXAC9IVRw",
}

for label, vid_id in check_videos.items():
    print(f"\n{label} ({vid_id}):")
    try:
        transcript_list = ytt_api.list(vid_id)
        for t in transcript_list:
            print(f"  {t.language} ({t.language_code}) — generated: {t.is_generated}")
    except TranscriptsDisabled:
        print(f"  Transcripts completely disabled for this video")
    except Exception as e:
        print(f"  Error: {type(e).__name__}: {e}")


bad_bunny (g3jCAyPai2Y):
  Transcripts completely disabled for this video

me_at_zoo (jNQXAC9IVRw):
  English (en) — generated: False
  German (de) — generated: False


In [17]:
# Better test candidates
more_tests = {
    "long_podcast":     "e-gwvmhyU7A",   # Lex Fridman podcast, ~3hrs
    "ted_talk":         "8jPQjjsBbIc",    # TED talk, ~15 min, likely auto-generated
    "random_vlog":      "BaW_jenozKc",    # Jeffree Star, likely auto-generated
}

for label, vid_id in more_tests.items():
    print(f"\n{label} ({vid_id}):")
    try:
        transcript_list = ytt_api.list(vid_id)
        for t in transcript_list:
            gen = "AUTO" if t.is_generated else "MANUAL"
            print(f"  [{gen}] {t.language} ({t.language_code})")
        
        # Fetch default and check size
        fetched = ytt_api.fetch(vid_id)
        last = fetched.snippets[-1]
        duration = last.start + last.duration
        print(f"  → Fetched: {len(fetched.snippets)} snippets, {duration:.0f}s ({duration/60:.1f} min), generated: {fetched.is_generated}")
    except Exception as e:
        print(f"  Error: {type(e).__name__}: {e}")


long_podcast (e-gwvmhyU7A):
  [MANUAL] English - Default (en)
  [AUTO] English (auto-generated) (en)
  → Fetched: 4371 snippets, 10927s (182.1 min), generated: False

ted_talk (8jPQjjsBbIc):
  [MANUAL] Arabic (ar)
  [MANUAL] Bulgarian (bg)
  [MANUAL] Burmese (my)
  [MANUAL] Central Kurdish (ckb)
  [MANUAL] Chinese (China) (zh-CN)
  [MANUAL] Chinese (Taiwan) (zh-TW)
  [MANUAL] Croatian (hr)
  [MANUAL] Czech (cs)
  [MANUAL] Dutch (nl)
  [MANUAL] English (en)
  [MANUAL] Esperanto (eo)
  [MANUAL] French (fr)
  [MANUAL] French (Canada) (fr-CA)
  [MANUAL] Galician (gl)
  [MANUAL] German (de)
  [MANUAL] Greek (el)
  [MANUAL] Hebrew (iw)
  [MANUAL] Hindi (hi)
  [MANUAL] Hungarian (hu)
  [MANUAL] Indonesian (id)
  [MANUAL] Italian (it)
  [MANUAL] Japanese (ja)
  [MANUAL] Korean (ko)
  [MANUAL] Latvian (lv)
  [MANUAL] Lithuanian (lt)
  [MANUAL] Marathi (mr)
  [MANUAL] Persian (fa)
  [MANUAL] Polish (pl)
  [MANUAL] Portuguese (Brazil) (pt-BR)
  [MANUAL] Portuguese (Portugal) (pt-PT)
  [MANUAL] R

In [18]:
# Fetch auto-generated transcript (Lex Fridman has both manual and auto)
vid_id = "e-gwvmhyU7A"

# Find the auto-generated transcript and fetch it
transcript_list = ytt_api.list(vid_id)
auto_transcript = transcript_list.find_generated_transcript(["en"])
fetched_auto = auto_transcript.fetch()

print(f"Language: {fetched_auto.language} ({fetched_auto.language_code})")
print(f"Auto-generated: {fetched_auto.is_generated}")
print(f"Snippets: {len(fetched_auto.snippets)}")

print(f"\n--- Compare manual vs auto (first 3 snippets) ---")
manual = ytt_api.fetch(vid_id)
for i in range(3):
    m = manual.snippets[i]
    a = fetched_auto.snippets[i]
    print(f"\n  Manual: {m.text[:80]!r}  [{m.start:.1f}s]")
    print(f"  Auto:   {a.text[:80]!r}  [{a.start:.1f}s]")

Language: English (auto-generated) (en)
Auto-generated: True
Snippets: 4502

--- Compare manual vs auto (first 3 snippets) ---

  Manual: '- Can you have a conversation with an AI'  [0.1s]
  Auto:   'can you have a conversation with an AI'  [0.1s]

  Manual: 'where it feels like you\ntalk to Einstein or Feynman'  [2.7s]
  Auto:   'where it feels like you talk to Einstein'  [2.7s]

  Manual: 'where you ask them a hard question,'  [7.3s]
  Auto:   'mhm or Fineman where you ask them a hard'  [5.2s]


## 3. Findings summary

**Data shape:**
- `ytt_api.fetch(video_id)` → `FetchedTranscript` with `.snippets`, `.language`, `.language_code`, `.is_generated`, `.video_id`
- Each snippet: `.text` (str), `.start` (float, seconds), `.duration` (float, seconds)
- Text can contain `\n` within a snippet
- Timestamps are floats in seconds, small gaps between snippets (0.08–1s typical), larger gaps for silence

**API behaviour:**
- `.fetch()` prefers manual captions over auto-generated by default — this is what we want
- `.list()` shows all available transcripts (manual + auto + translations)
- Auto-generated: no punctuation, filler words, different segment boundaries
- Video duration ≈ last snippet's `start + duration` — good enough for our 60-min cap check

**Errors observed:**
- `TranscriptsDisabled` — video has captions turned off entirely (Bad Bunny example)
- `VideoUnavailable` — video removed or region-locked
- `NoTranscriptFound` — not triggered yet but imported for safety

**Decisions for production:**
- Use default `.fetch()` (manual preferred) — don't force auto-generated
- Estimate duration from last snippet, not from a separate API call
- Handle all three error types gracefully
- Convert snippets to plain dicts for downstream (chunking doesn't need the custom objects)

## 4. Production function

In [ ]:
# @export src/transcript.py
import os
import json
import urllib.request
import urllib.error
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable,
    IpBlocked,
    RequestBlocked,
)


def _fetch_via_proxy(video_id: str) -> dict:
    """Fetch transcript via Cloudflare Worker proxy (fallback for IP blocks)."""
    proxy_url = os.getenv("TRANSCRIPT_PROXY_URL")
    proxy_secret = os.getenv("TRANSCRIPT_PROXY_SECRET", "")
    if not proxy_url:
        raise ValueError("YouTube is blocking requests and no transcript proxy is configured.")

    payload = json.dumps({"video_id": video_id}).encode()
    headers = {"Content-Type": "application/json"}
    if proxy_secret:
        headers["Authorization"] = f"Bearer {proxy_secret}"

    req = urllib.request.Request(proxy_url, data=payload, headers=headers, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode())
    except urllib.error.HTTPError as e:
        body = e.read().decode() if e.fp else ""
        try:
            err_data = json.loads(body)
            msg = err_data.get("error", body[:100])
        except Exception:
            msg = body[:100] or f"HTTP {e.code}"
        raise ValueError(f"Transcript proxy error: {msg}")
    except Exception as e:
        raise ValueError(f"Transcript proxy unreachable: {type(e).__name__}")

    return {
        "video_id": data["video_id"],
        "language": data.get("language", "unknown"),
        "is_generated": data.get("is_generated", True),
        "snippets": data["snippets"],
        "duration_seconds": data["duration_seconds"],
    }


def fetch_transcript(video_id: str) -> dict:
    """Fetch transcript for a YouTube video.
    
    Tries the youtube_transcript_api first. If blocked by YouTube IP filtering,
    falls back to a Cloudflare Worker proxy (if TRANSCRIPT_PROXY_URL is set).
    
    Returns dict with:
        - video_id: str
        - language: str
        - is_generated: bool
        - snippets: list of {text, start, duration}
        - duration_seconds: float (estimated from last snippet)
    
    Raises:
        ValueError: if transcript is unavailable
    """
    ytt_api = YouTubeTranscriptApi()
    
    try:
        transcript = ytt_api.fetch(video_id)
    except TranscriptsDisabled:
        raise ValueError(f"Transcripts are disabled for video {video_id}")
    except NoTranscriptFound:
        raise ValueError(f"No transcript found for video {video_id}")
    except VideoUnavailable:
        raise ValueError(f"Video {video_id} is unavailable")
    except (IpBlocked, RequestBlocked):
        return _fetch_via_proxy(video_id)
    except Exception as e:
        raise ValueError(f"Could not fetch transcript for video {video_id}: {type(e).__name__}")
    
    snippets = [
        {"text": s.text, "start": s.start, "duration": s.duration}
        for s in transcript.snippets
    ]
    
    last = transcript.snippets[-1]
    duration_seconds = last.start + last.duration
    
    return {
        "video_id": video_id,
        "language": transcript.language,
        "is_generated": transcript.is_generated,
        "snippets": snippets,
        "duration_seconds": duration_seconds,
    }

In [20]:
# Test happy path
result = fetch_transcript("dQw4w9WgXcQ")
print(f"video_id: {result['video_id']}")
print(f"language: {result['language']}")
print(f"is_generated: {result['is_generated']}")
print(f"snippets: {len(result['snippets'])}")
print(f"duration: {result['duration_seconds']:.0f}s ({result['duration_seconds']/60:.1f} min)")
print(f"snippet type: {type(result['snippets'][0])}")
print(f"first snippet: {result['snippets'][0]}")

# Test error handling
print("\n--- Error cases ---")
for vid_id, label in [("g3jCAyPai2Y", "disabled"), ("BaW_jenozKc", "unavailable"), ("xxxxxxxxxxxx", "fake ID")]:
    try:
        fetch_transcript(vid_id)
        print(f"  {label}: ❌ should have raised")
    except ValueError as e:
        print(f"  {label}: ✅ {e}")
    except Exception as e:
        print(f"  {label}: ⚠️ unexpected: {type(e).__name__}: {e}")

video_id: dQw4w9WgXcQ
language: English
is_generated: False
snippets: 61
duration: 211s (3.5 min)
snippet type: <class 'dict'>
first snippet: {'text': '[♪♪♪]', 'start': 1.36, 'duration': 1.68}

--- Error cases ---
  disabled: ✅ Transcripts are disabled for video g3jCAyPai2Y
  unavailable: ✅ Video BaW_jenozKc is unavailable
  fake ID: ✅ Video xxxxxxxxxxxx is unavailable


## 5. URL parsing helper

Production will receive full YouTube URLs from the user, not raw video IDs. Need to extract the ID from various URL formats.

In [21]:
# @export src/transcript.py
import re


def extract_video_id(url: str) -> str:
    """Extract video ID from various YouTube URL formats.
    
    Supports:
        - https://www.youtube.com/watch?v=VIDEO_ID
        - https://youtu.be/VIDEO_ID
        - https://www.youtube.com/embed/VIDEO_ID
        - https://www.youtube.com/v/VIDEO_ID
        - Raw video ID (11 characters)
    
    Raises:
        ValueError: if no valid video ID can be extracted
    """
    # Already a raw video ID (11 chars, alphanumeric + _ -)
    if re.fullmatch(r"[A-Za-z0-9_-]{11}", url.strip()):
        return url.strip()
    
    patterns = [
        r"(?:youtube\.com/watch\?.*v=)([A-Za-z0-9_-]{11})",
        r"(?:youtu\.be/)([A-Za-z0-9_-]{11})",
        r"(?:youtube\.com/embed/)([A-Za-z0-9_-]{11})",
        r"(?:youtube\.com/v/)([A-Za-z0-9_-]{11})",
    ]
    
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    
    raise ValueError(f"Could not extract video ID from: {url}")

In [22]:
test_urls = [
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ",
    "https://youtu.be/dQw4w9WgXcQ",
    "https://www.youtube.com/embed/dQw4w9WgXcQ",
    "https://www.youtube.com/v/dQw4w9WgXcQ",
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ&t=120",
    "dQw4w9WgXcQ",
    "https://www.youtube.com/watch?list=PLrAXtmErZgOeiKm4sgNOknGvNjby9efdf&v=dQw4w9WgXcQ",
]

print("--- Valid URLs ---")
for url in test_urls:
    vid_id = extract_video_id(url)
    print(f"  ✅ {vid_id} ← {url[:60]}")

print("\n--- Invalid URLs ---")
for url in ["https://google.com", "not-a-url", "https://youtube.com/channel/abc"]:
    try:
        extract_video_id(url)
        print(f"  ❌ should have raised for: {url}")
    except ValueError as e:
        print(f"  ✅ caught: {e}")

--- Valid URLs ---
  ✅ dQw4w9WgXcQ ← https://www.youtube.com/watch?v=dQw4w9WgXcQ
  ✅ dQw4w9WgXcQ ← https://youtu.be/dQw4w9WgXcQ
  ✅ dQw4w9WgXcQ ← https://www.youtube.com/embed/dQw4w9WgXcQ
  ✅ dQw4w9WgXcQ ← https://www.youtube.com/v/dQw4w9WgXcQ
  ✅ dQw4w9WgXcQ ← https://www.youtube.com/watch?v=dQw4w9WgXcQ&t=120
  ✅ dQw4w9WgXcQ ← dQw4w9WgXcQ
  ✅ dQw4w9WgXcQ ← https://www.youtube.com/watch?list=PLrAXtmErZgOeiKm4sgNOknGv

--- Invalid URLs ---
  ✅ caught: Could not extract video ID from: https://google.com
  ✅ caught: Could not extract video ID from: not-a-url
  ✅ caught: Could not extract video ID from: https://youtube.com/channel/abc


## Summary

**What we built:**
- `extract_video_id(url)` — parses 5 URL formats + raw ID, raises `ValueError` on invalid input
- `fetch_transcript(video_id)` — returns clean dict with snippets, language, duration; raises `ValueError` on all failure modes

**Production cells tagged:** 2 cells → `src/transcript.py`

**Key decisions:**
- Default `.fetch()` prefers manual captions — no need to specify
- Duration estimated from last snippet (`start + duration`) — good enough for 60-min cap
- Errors unified to `ValueError` with descriptive messages — keeps consumer code simple
- Snippets converted to plain dicts — decouples from library internals

**Edge cases tested:**
| Case | Video | Result |
|---|---|---|
| Short, manual captions | dQw4w9WgXcQ (3.5 min) | ✅ 61 snippets |
| Long video | e-gwvmhyU7A (182 min) | ✅ 4371 snippets |
| Auto-generated available | e-gwvmhyU7A | ✅ defaults to manual |
| Transcripts disabled | g3jCAyPai2Y | ✅ ValueError |
| Video unavailable | BaW_jenozKc | ✅ ValueError |
| Fake video ID | xxxxxxxxxxxx | ✅ ValueError |
| Multiple URL formats |

In [26]:
# Save transcript data for use in other notebooks (avoids repeated API calls + rate limits)
import json

test_data = {}
for vid_id, label in [("dQw4w9WgXcQ", "short"), ("e-gwvmhyU7A", "long")]:
    t = ytt_api.fetch(vid_id)
    test_data[label] = {
        "video_id": vid_id,
        "language": t.language,
        "is_generated": t.is_generated,
        "snippets": [{"text": s.text, "start": s.start, "duration": s.duration} for s in t.snippets],
    }

with open("../data/test_transcripts.json", "w") as f:
    json.dump(test_data, f, indent=2)

print(f"Saved: short ({len(test_data['short']['snippets'])} snippets), long ({len(test_data['long']['snippets'])} snippets)")

Saved: short (61 snippets), long (4371 snippets)
